# 🚀 AIC 2026: End-to-End Dual-GPU (2× T4) Production Pipeline (Scene-Adaptive Sampling)

This notebook implements the complete **AI Challenge (AIC) 2026** multi-modal retrieval pipeline:
1. **GPU 0**: SigLIP-SO400M @ 384px **scene-adaptive shot sampling** (eliminates redundant static frames)
2. **GPU 1**: Whisper large-v3 Vietnamese audio transcription & on-screen OCR
3. **Unified Indexing**: Scalable FAISS FlatIP vector index (~120k–400k frames) + Multi-modal BM25 lexical index
4. **Stage 2 Exact Localizer**: 30fps dense video decode around candidate timestamps (KIS & Q&A)
5. **TRAKE Stage 1**: DP-aligned video retrieval via scene index (Stage 2 VLM localization = future work)
6. **Submission Engine**: 100-rank portfolio optimization for competition metric $\frac{1}{5}\sum R@k$


In [ ]:
!git clone https://github.com/TotallyNotMinh/aic2026.git

In [ ]:
!git pull

In [ ]:
# 1. Verify Dual GPU Hardware (2x NVIDIA T4) & Setup Working Directory
import torch, os, sys

# Navigate into cloned repository if present
REPO_DIR = '/kaggle/working/aic2026'
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f'Active working directory set to: {os.getcwd()}')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} CUDA GPUs:')
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)')

assert num_gpus >= 1, 'Please enable GPU accelerator in Kaggle Settings (2x T4 recommended)!'

In [ ]:
# 2. Install Required Dependencies
!pip install -q open-clip-torch transformers faster-whisper openai-whisper faiss-cpu rank-bm25 deep-translator opencv-python easyocr fiftyone

In [ ]:
!pip install -q "pillow<11.0.0"


In [ ]:
# 3. Symlink / Prepare Data Directories from Kaggle Input into aic2026/data
import os, glob

target_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(target_dir, exist_ok=True)

if os.path.exists('/kaggle/input'):
    print(f'Detected Kaggle environment. Linking input datasets into {target_dir}...')
    for p in glob.glob('/kaggle/input/**/Videos_*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/Keyframes_*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*media-info*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*map-keyframes*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*-aic25-b1', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)

print(f'Linked data contents ({target_dir}):', sorted(os.listdir(target_dir)))

### ⚡ Step 4: Parallel Dual-GPU Feature Extraction (Scene-Adaptive Video + Whisper ASR)

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor
import torch
from scripts.extract_siglip_features import extract_all_siglip_features
from scripts.extract_whisper_asr import extract_all_whisper_asr
from scripts.extract_ocr import extract_all_ocr
print('[Pipeline] Starting In-Process Multi-GPU Feature Extraction...')
t_start = time.time()
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f'Detected {num_gpus} GPU(s).')

In [ ]:
# --- Stage 1: SigLIP Vision Extraction ---
print('\n🚀 Stage 1/3: Extracting SigLIP Vision Features...')
if num_gpus >= 2:
    with ThreadPoolExecutor(max_workers=2) as executor:
        f0 = executor.submit(extract_all_siglip_features, device='cuda:0', batch_size=256, sample_interval_sec=1.5, num_shards=2, shard_id=0)
        f1 = executor.submit(extract_all_siglip_features, device='cuda:1', batch_size=256, sample_interval_sec=1.5, num_shards=2, shard_id=1)
        f0.result()
        f1.result()
else:
    extract_all_siglip_features(device='cuda:0', batch_size=256, sample_interval_sec=1.5, num_shards=1, shard_id=0)
print('✅ Stage 1 complete: SigLIP features extracted.')


In [ ]:
import os, sys, gc, glob, json, time, subprocess, shutil, queue, threading
from concurrent.futures import ThreadPoolExecutor
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x
import torch

# 1. Clean VRAM & Enable Tensor Core Accelerations
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# 2. Parallel CPU Audio Extraction Worker (Background Prefetch)
def extract_single_audio(vid_path, audio_dir, kaggle_audio_dir):
    vid_name = os.path.splitext(os.path.basename(vid_path))[0]
    out_wav = os.path.join(audio_dir, f"{vid_name}.wav")
    kaggle_wav = os.path.join(kaggle_audio_dir, f"{vid_name}.wav") if kaggle_audio_dir else None

    # Check if already extracted
    if os.path.exists(out_wav) and os.path.getsize(out_wav) > 1000:
        return vid_name, out_wav
    if kaggle_wav and os.path.exists(kaggle_wav) and os.path.getsize(kaggle_wav) > 1000:
        return vid_name, kaggle_wav

    tmp_wav = f"{out_wav}.tmp.{os.getpid()}_{threading.get_ident()}.wav"
    try:
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", vid_path,
            "-vn", "-ac", "1", "-ar", "16000",
            "-c:a", "pcm_s16le",
            "-f", "wav", tmp_wav
        ]
        res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=120)
        if res.returncode == 0 and os.path.exists(tmp_wav) and os.path.getsize(tmp_wav) > 100:
            os.replace(tmp_wav, out_wav)
            if kaggle_wav and os.path.exists(kaggle_audio_dir):
                try:
                    shutil.copy2(out_wav, kaggle_wav)
                except Exception:
                    pass
            return vid_name, out_wav
    except Exception:
        pass
    finally:
        if os.path.exists(tmp_wav):
            try:
                os.remove(tmp_wav)
            except Exception:
                pass
    
    # Fallback to direct video file path if no audio stream
    return vid_name, vid_path

# 3. High-Throughput Faster-Whisper / PhoWhisper Worker with Batched VAD
def run_phowhisper_worker(gpu_id, shard_id, num_shards, videos_root="data", output_dir="cache/asr_transcripts", audio_dir="cache/audio_extracted"):
    if torch.cuda.is_available():
        torch.cuda.set_device(gpu_id)

    kaggle_out = "/kaggle/working/cache/asr_transcripts"
    kaggle_audio = "/kaggle/working/cache/audio_extracted"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(audio_dir, exist_ok=True)
    try:
        os.makedirs(kaggle_out, exist_ok=True)
        os.makedirs(kaggle_audio, exist_ok=True)
    except Exception:
        pass
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if torch.cuda.is_available() else "int8"
    
    # Try loading Faster-Whisper CTranslate2 PhoWhisper-medium (4-6x faster than standard transformers)
    model_loaded = None
    batched_model = None
    transcriber_pipe = None
    
    ct2_model_candidates = [
        "quocphu/PhoWhisper-ct2-FasterWhisper/PhoWhisper-medium-ct2-fasterWhisper",
        "thoaibuiic/PhoWhisper-medium-ct2",
        "vinai/phowhisper-medium",
        "medium"
    ]
    
    try:
        from faster_whisper import WhisperModel, BatchedInferencePipeline
        for m_candidate in ct2_model_candidates:
            try:
                print(f"[GPU {gpu_id}] Attempting to load faster-whisper model '{m_candidate}' on {device}:{gpu_id} ({compute_type})...")
                model_loaded = WhisperModel(
                    m_candidate,
                    device=device,
                    device_index=gpu_id,
                    compute_type=compute_type,
                    num_workers=1,
                    cpu_threads=1,  # Set to 1 thread to avoid CPU bottleneck
                )
                batched_model = BatchedInferencePipeline(model=model_loaded)
                print(f"[GPU {gpu_id}] ✅ faster-whisper '{m_candidate}' loaded successfully with BatchedInferencePipeline!")
                break
            except Exception as load_err:
                continue
    except Exception as import_err:
        pass
        
    # Fallback to Hugging Face transformers pipeline if faster-whisper ct2 is unavailable
    if batched_model is None:
        print(f"[GPU {gpu_id}] Falling back to HuggingFace pipeline with 'vinai/phowhisper-medium'...")
        from transformers import pipeline
        transcriber_pipe = pipeline(
            "automatic-speech-recognition",
            model="vinai/phowhisper-medium",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device=f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu",
            chunk_length_s=30,
            stride_length_s=6,
            return_timestamps=True,
            model_kwargs={"attn_implementation": "sdpa"} if torch.cuda.is_available() else {}
        )
        print(f"[GPU {gpu_id}] HuggingFace PhoWhisper pipeline ready!")
    
    # Comprehensive discovery for the complete 120-hour dataset (no skipping)
    candidate_paths = (
        glob.glob(os.path.join(videos_root, "Videos_L*", "video", "*.mp4")) +
        glob.glob(os.path.join(videos_root, "Videos_L*", "*.mp4")) +
        glob.glob(os.path.join(videos_root, "**", "*.mp4"), recursive=True) +
        glob.glob("/kaggle/input/**/Videos_L*/**/*.mp4", recursive=True) +
        glob.glob("/kaggle/input/**/video/*.mp4", recursive=True) +
        glob.glob("/kaggle/input/**/*.mp4", recursive=True)
    )
    
    # Deduplicate videos by video_id
    video_map = {}
    for p in candidate_paths:
        v_name = os.path.splitext(os.path.basename(p))[0]
        if v_name not in video_map and os.path.isfile(p):
            video_map[v_name] = p
            
    all_videos = sorted(video_map.values())
    shard_videos = [f for idx, f in enumerate(all_videos) if idx % num_shards == shard_id]
    print(f"[GPU {gpu_id}] Total dataset: {len(all_videos)} videos | Shard {shard_id}: {len(shard_videos)} videos.")
    
    # Background Asynchronous Audio Prefetch Queue (1 CPU worker per GPU to keep CPU 100% responsive)
    prefetch_queue = queue.Queue(maxsize=16)
    stop_signal = object()

    def cpu_prefetch_producer():
        with ThreadPoolExecutor(max_workers=1) as cpu_pool:
            futures = []
            for v_path in shard_videos:
                fut = cpu_pool.submit(extract_single_audio, v_path, audio_dir, kaggle_audio)
                futures.append((v_path, fut))
            
            for v_path, fut in futures:
                try:
                    v_name, audio_path = fut.result()
                    prefetch_queue.put((v_path, v_name, audio_path))
                except Exception:
                    v_name = os.path.splitext(os.path.basename(v_path))[0]
                    prefetch_queue.put((v_path, v_name, v_path))
        prefetch_queue.put(stop_signal)

    producer_thread = threading.Thread(target=cpu_prefetch_producer, daemon=True)
    producer_thread.start()

    # Dynamic Batch Size Tracker (Starts at full VRAM capacity, auto-halves on OOM)
    current_batch_size = 24 if torch.cuda.is_available() else 4

    pbar = tqdm(total=len(shard_videos), desc=f"PhoWhisper GPU {gpu_id}", position=gpu_id, leave=True, dynamic_ncols=True)
    while True:
        item = prefetch_queue.get()
        if item is stop_signal:
            break
        
        vid_path, vid_name, audio_path = item
        out_json = os.path.join(output_dir, f"{vid_name}.json")
        kaggle_json = os.path.join(kaggle_out, f"{vid_name}.json")
        
        # Check if already completed (instant skip)
        if os.path.exists(out_json) and os.path.getsize(out_json) > 10:
            pbar.update(1)
            continue
        if os.path.exists(kaggle_json) and os.path.getsize(kaggle_json) > 10:
            try:
                shutil.copy2(kaggle_json, out_json)
            except Exception:
                pass
            pbar.update(1)
            continue

        # Transcribe with Auto-OOM Recovery & Low-Threshold VAD Parameters
        segments = []
        bs = current_batch_size
        while bs >= 2:
            try:
                if batched_model is not None:
                    # faster-whisper Batched Pipeline
                    segments_gen, _ = batched_model.transcribe(
                        audio_path,
                        batch_size=bs,
                        language="vi",
                        vad_filter=True,
                        vad_parameters=dict(
                            min_silence_duration_ms=500,
                        ),
                        no_speech_threshold=0.3,
                        log_prob_threshold=-1.0,
                        temperature=0.0,
                    )
                    for seg in segments_gen:
                        txt = (seg.text or "").strip()
                        if txt:
                            segments.append({
                                "video_id": vid_name,
                                "start_sec": round(float(seg.start), 2),
                                "end_sec": round(float(seg.end), 2),
                                "start_frame": int(round(seg.start * 25.0)),
                                "end_frame": int(round(seg.end * 25.0)),
                                "text": txt,
                            })
                else:
                    # Transformers Pipeline Fallback
                    result = transcriber_pipe(
                        audio_path,
                        generate_kwargs={
                            "language": "vi",
                            "task": "transcribe",
                            "temperature": 0.0,
                            "no_speech_threshold": 0.3,
                            "logprob_threshold": -1.0,
                        },
                        return_timestamps=True,
                        batch_size=bs
                    )
                    chunks = result.get("chunks", []) if isinstance(result, dict) else []
                    for chunk in chunks:
                        txt = (chunk.get("text") or "").strip()
                        ts = chunk.get("timestamp") or (0.0, 0.0)
                        st = float(ts[0]) if ts[0] is not None else 0.0
                        et = float(ts[1]) if ts[1] is not None else round(st + 3.0, 2)
                        if txt:
                            segments.append({
                                "video_id": vid_name,
                                "start_sec": round(st, 2),
                                "end_sec": round(et, 2),
                                "start_frame": int(round(st * 25.0)),
                                "end_frame": int(round(et * 25.0)),
                                "text": txt,
                            })
                current_batch_size = bs
                break
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                err_str = str(e).lower()
                if "out of memory" in err_str or "cuda" in err_str:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    bs = max(2, bs // 2)
                    print(f"[GPU {gpu_id}] ⚠️ VRAM saturated on {vid_name}! Reducing batch_size to {bs}...")
                else:
                    print(f"[GPU {gpu_id}] Non-OOM error on {vid_name}: {e}")
                    segments = []
                    break
            except Exception as e:
                print(f"[GPU {gpu_id}] Error processing {vid_name}: {e}")
                segments = []
                break

        # Save JSON output immediately per video (atomic write)
        tmp_json = f"{out_json}.tmp.{os.getpid()}_{threading.get_ident()}"
        with open(tmp_json, "w", encoding="utf-8") as f:
            json.dump(segments, f, indent=2, ensure_ascii=False)
        os.replace(tmp_json, out_json)

        # Automatically mirror completed transcript to Kaggle storage
        try:
            if os.path.exists(kaggle_out):
                shutil.copy2(out_json, kaggle_json)
        except Exception:
            pass

        pbar.update(1)

    pbar.close()
    print(f"✅ GPU {gpu_id} finished all videos!")

# 4. Launch Dual-GPU Parallel Execution (2x T4)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"🚀 Launching PhoWhisper Medium across {num_gpus} GPU(s) with faster-whisper & parallel CPU prefetch...")
t0 = time.time()

if num_gpus >= 2:
    with ThreadPoolExecutor(max_workers=2) as executor:
        f0 = executor.submit(run_phowhisper_worker, gpu_id=0, shard_id=0, num_shards=2)
        f1 = executor.submit(run_phowhisper_worker, gpu_id=1, shard_id=1, num_shards=2)
        f0.result()
        f1.result()
else:
    run_phowhisper_worker(gpu_id=0, shard_id=0, num_shards=1)

print(f"🎉 100% of 120hr dataset transcribed in {(time.time() - t0)/60:.1f} minutes!")


In [ ]:
import os, sys, gc, glob, json, time
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import torch
import cv2

# 1. Clean VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Scene-Aligned OCR Worker
def run_scene_ocr_worker(gpu_id, shard_id, num_shards, videos_root="data", meta_dir="cache/siglip_meta", output_dir="cache/ocr_text"):
    os.makedirs(output_dir, exist_ok=True)
    if torch.cuda.is_available():
        torch.cuda.set_device(gpu_id)
        
    import easyocr
    print(f"[GPU {gpu_id}] Loading EasyOCR (vi + en) on cuda:{gpu_id}...")
    reader = easyocr.Reader(["vi", "en"], gpu=torch.cuda.is_available())
    print(f"[GPU {gpu_id}] EasyOCR loaded!")
    
    all_videos = sorted(glob.glob(os.path.join(videos_root, "Videos_L*", "video", "*.mp4")))
    shard_videos = [f for idx, f in enumerate(all_videos) if idx % num_shards == shard_id]
    
    for vid_path in tqdm(shard_videos, desc=f"Scene OCR GPU {gpu_id}"):
        vid_name = os.path.splitext(os.path.basename(vid_path))[0]
        out_json = os.path.join(output_dir, f"{vid_name}.json")
        meta_json = os.path.join(meta_dir, f"{vid_name}.json")
        
        if os.path.exists(out_json):
            continue
            
        # Get exact scene keyframe indices from Stage 1 metadata
        keyframe_indices = []
        if os.path.exists(meta_json):
            with open(meta_json, "r", encoding="utf-8") as f:
                meta_list = json.load(f)
                keyframe_indices = [item["frame_idx"] for item in meta_list if "frame_idx" in item]

        if not keyframe_indices:
            # Fallback if metadata missing
            keyframe_indices = list(range(0, 30 * 600, 90))
            
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            continue
            
        ocr_results = {}
        try:
            for f_idx in keyframe_indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
                ret, frame = cap.read()
                if not ret:
                    continue

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                try:
                    results = reader.readtext(rgb)
                    texts = [r[1] for r in results if len(r) > 2 and float(r[2]) > 0.35]
                    if texts:
                        ocr_results[f"f_{f_idx}"] = " ".join(texts)
                except Exception:
                    pass
        finally:
            cap.release()
            
        # Atomic write
        tmp_json = f"{out_json}.tmp.{os.getpid()}"
        with open(tmp_json, "w", encoding="utf-8") as f:
            json.dump(ocr_results, f, indent=2, ensure_ascii=False)
        os.replace(tmp_json, out_json)

    print(f"✅ GPU {gpu_id} completed Scene-Aligned OCR!")

# 3. Launch Dual-GPU Parallel Execution
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"🚀 Starting Scene-Aligned Video OCR across {num_gpus} GPU(s)...")
t0 = time.time()

if num_gpus >= 2:
    with ThreadPoolExecutor(max_workers=2) as executor:
        f0 = executor.submit(run_scene_ocr_worker, gpu_id=0, shard_id=0, num_shards=2)
        f1 = executor.submit(run_scene_ocr_worker, gpu_id=1, shard_id=1, num_shards=2)
        f0.result()
        f1.result()
else:
    run_scene_ocr_worker(gpu_id=0, shard_id=0, num_shards=1)

print(f"🎉 100% Scene OCR extracted in {(time.time() - t0)/60:.1f} minutes!")


### 🏗️ Step 5: Build Scalable FAISS & Multi-Modal BM25 Indices

In [ ]:
# Build production FAISS index & unified lexical BM25 index
!python scripts/build_faiss_index.py

from src.index.metadata_indexer import MetadataIndexer
meta_idx = MetadataIndexer().build_and_cache(force=True)
print('Unified indices successfully built!')

### 🔍 Step 6: Interactive Multi-Modal Retrieval with Stage 2 Dense Localization

In [1]:
import os, sys, gc, glob, json, time, queue
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import torch
import cv2

# 1. Clean VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Build single shared dynamic work queue of only uncompleted videos
output_dir = "cache/ocr_text"
os.makedirs(output_dir, exist_ok=True)

all_videos = sorted(glob.glob("data/Videos_L*/video/*.mp4"))
work_queue = queue.Queue()

for v in all_videos:
    v_name = os.path.splitext(os.path.basename(v))[0]
    out_json = os.path.join(output_dir, f"{v_name}.json")
    if not os.path.exists(out_json):
        work_queue.put(v)

total_pending = work_queue.qsize()
print(f"📦 Total uncompleted videos in shared queue: {total_pending} / {len(all_videos)}")

# 3. Dynamic Worker (Workers steal next video from shared queue as soon as they are free)
def run_dynamic_ocr_worker(gpu_id, q, pbar):
    if torch.cuda.is_available():
        torch.cuda.set_device(gpu_id)
        
    import easyocr, av
    reader = easyocr.Reader(["vi", "en"], gpu=torch.cuda.is_available())
    
    while True:
        try:
            vid_path = q.get_nowait()
        except queue.Empty:
            break
            
        vid_name = os.path.splitext(os.path.basename(vid_path))[0]
        out_json = os.path.join(output_dir, f"{vid_name}.json")
        
        ocr_results = {}
        try:
            container = av.open(vid_path)
            stream = container.streams.video[0]
            stream.codec_context.skip_frame = "NONKEY"
            
            fps = float(stream.average_rate) if stream.average_rate else 30.0
            time_base = float(stream.time_base) if stream.time_base else (1.0 / fps)
            
            last_processed_pts = -999.0
            
            for packet in container.demux(stream):
                for frame in packet.decode():
                    pts_sec = float(frame.pts * time_base) if frame.pts is not None else 0.0

                    # Ensure minimum 2.5s gap between OCR checks (caps long 25-min videos at ~100 frames)
                    if (pts_sec - last_processed_pts) < 2.5:
                        continue

                    arr = frame.to_ndarray(format="rgb24")

                    # 0.001s fast edge check
                    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
                    if cv2.Laplacian(gray, cv2.CV_16S).var() < 80:
                        continue

                    try:
                        results = reader.readtext(arr, canvas_size=800, mag_ratio=1.0)
                        texts = [r[1] for r in results if len(r) > 2 and float(r[2]) > 0.35]
                        if texts:
                            frame_idx = int(round(pts_sec * fps))
                            ocr_results[f"f_{frame_idx}"] = " ".join(texts)
                            last_processed_pts = pts_sec
                    except Exception:
                        pass
            container.close()
        except Exception:
            pass
            
        tmp_json = f"{out_json}.tmp.{os.getpid()}"
        with open(tmp_json, "w", encoding="utf-8") as f:
            json.dump(ocr_results, f, indent=2, ensure_ascii=False)
        os.replace(tmp_json, out_json)
        
        q.task_done()
        pbar.update(1)

# 4. Launch Dynamic Dual-GPU Execution
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"🚀 Starting Dynamic Work-Stealing OCR on {num_gpus} GPU(s)...")
t0 = time.time()

with tqdm(total=total_pending, desc="Dynamic Dual-GPU OCR") as pbar:
    if num_gpus >= 2:
        with ThreadPoolExecutor(max_workers=2) as executor:
            f0 = executor.submit(run_dynamic_ocr_worker, 0, work_queue, pbar)
            f1 = executor.submit(run_dynamic_ocr_worker, 1, work_queue, pbar)
            f0.result()
            f1.result()
    else:
        run_dynamic_ocr_worker(0, work_queue, pbar)

print(f"🎉 100% OCR completed in {(time.time() - t0)/60:.1f} minutes!")


📦 Total uncompleted videos in shared queue: 365 / 873
🚀 Starting Dynamic Work-Stealing OCR on 2 GPU(s)...


Dynamic Dual-GPU OCR: 100%|██████████| 365/365 [28:54<00:00,  4.75s/it]

🎉 100% OCR completed in 28.9 minutes!


### 📦 Step 7: Export Official Competition Submissions (Exact 100 Rows)

In [ ]:
# Generate official 100-row submission CSV and package bundle ZIP
sample_preds = [{'video_id': r[0]['video_id'], 'frame_idx': r[0]['frame_idx']} for r in results]
lines = sub_gen.format_kis_submission('query_01', sample_preds)
sub_gen.save_submission_file('query_01', lines)
zip_path = sub_gen.package_submission_zip('AIC2026_Submission_Bundle.zip')
print(f'Official Submission Package Ready: {zip_path} (Contains {len(lines)} rows in query_01.csv)')

### 📥 Step 8: Package & Download All Encoded Features (Visual, Audio ASR, OCR & FAISS Indices)

Compresses all generated `.npy` visual embeddings, Whisper transcripts, OCR text, and FAISS indices into a single downloadable ZIP archive in `/kaggle/working/`.

In [ ]:
import os, zipfile
from IPython.display import FileLink

zip_filename = 'aic2026_features_and_cache.zip'
zip_path = os.path.join('/kaggle/working' if os.path.exists('/kaggle/working') else '.', zip_filename)

print(f'[Export] Packaging cache directory into {zip_path}...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('cache'):
        for file in files:
            if file.endswith(('.npy', '.json', '.index', '.pkl')) and '.tmp.' not in file:
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(abs_path, '.')
                zf.write(abs_path, arcname=rel_path)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'✅ All features successfully packaged! Total Archive Size: {zip_size_mb:.2f} MB')
print(f'Artifact saved at: {zip_path}')
FileLink(zip_filename)

In [ ]:
import os
%cd /kaggle/working/aic2026

# Kiểm tra số lượng file đã trích xuất
npy_count = len(os.listdir('cache/siglip_features'))
json_count = len(os.listdir('cache/siglip_meta'))
print(f"✅ Đã bảo toàn: {npy_count} file .npy và {json_count} file .json!")

# Nén để tải về
!zip -r -q /kaggle/working/siglip_cache.zip cache/siglip_features cache/siglip_meta
print("📦 Đã nén xong thành /kaggle/working/siglip_cache.zip!")


In [ ]:
if os.path.exists('cache/asr_transcripts'):
    files = os.listdir('cache/asr_transcripts')
    json_files = [f for f in files if f.endswith('.json')]
    print(f"✅ Found {len(json_files)} ASR transcript files.")
else:
    print("❌ Directory cache/asr_transcripts not found!")

# 2. Compress into a single zip file in /kaggle/working/
!zip -r -q /kaggle/working/asr_transcripts.zip cache/asr_transcripts
print("📦 Successfully zipped to /kaggle/working/asr_transcripts.zip!")
print("👉 You can now download 'asr_transcripts.zip' from the Output section in the right sidebar.")


In [5]:
import os
import shutil
import zipfile
from IPython.display import FileLink, display

# 1. Zip ocr_text directly into /kaggle/working/ for easy 1-click download
src_dir = "/kaggle/working/cache/ocr_text"
zip_path = "/kaggle/working/ocr_text.zip"

print(f"📦 Zipping {src_dir} into {zip_path}...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(src_dir):
        for f in files:
            full_p = os.path.join(root, f)
            arcname = os.path.basename(f)
            zf.write(full_p, arcname=arcname)

# 2. Also copy the unzipped folder directly to /kaggle/working/ocr_text if needed
dst_dir = "/kaggle/working/ocr_text"
if not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)

print(f"✅ Done! Created /kaggle/working/ocr_text.zip ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)")
print("📁 Moved folder to /kaggle/working/ocr_text")


📦 Zipping /kaggle/working/cache/ocr_text into /kaggle/working/ocr_text.zip...
✅ Done! Created /kaggle/working/ocr_text.zip (0.62 MB)
📁 Moved folder to /kaggle/working/ocr_text
